# 第 7 章实验 · NLP 与 LLM：从 one-hot 到注意力、温度采样与最小 RAG

> 对应正文：[07-自然语言处理与LLM](../docs/4-专业方向/07-自然语言处理与LLM/README.md) ·
> [01 词向量与 embedding](../docs/4-专业方向/07-自然语言处理与LLM/01-词向量与embedding.md) ·
> [02 Transformer 与注意力机制](../docs/4-专业方向/07-自然语言处理与LLM/02-Transformer与注意力机制.md) ·
> [04 大语言模型 LLM 全景](../docs/4-专业方向/07-自然语言处理与LLM/04-大语言模型LLM全景.md) ·
> [06 RAG 与 Agent](../docs/4-专业方向/07-自然语言处理与LLM/06-RAG与Agent.md)

**环境**：实验 1~5、7 只需 `numpy / matplotlib / scikit-learn`，全部离线可跑（核心机制全部用 numpy 手写复刻）；
实验 6 的真实模型生成需要 `pip install transformers torch` 且首次联网下载模型（约 500MB），该 cell 仅语法校验并注明预期。
数据与向量全部由代码构造、随机种子固定——从上到下完整执行即复现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

np.random.seed(0)   # 固定随机种子，全 notebook 结果可复现

# 中文字体：Windows 优先微软雅黑/黑体，Mac/Linux 自动回退到检测到的字体
_available = {f.name for f in font_manager.fontManager.ttflist}
_cjk = [f for f in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
                    "PingFang SC", "WenQuanYi Micro Hei"] if f in _available]
plt.rcParams["font.sans-serif"] = _cjk + plt.rcParams["font.sans-serif"]
plt.rcParams["axes.unicode_minus"] = False    # 让负号正常显示

def softmax_rows(m, T=1.0):
    """数值稳定的逐行 softmax（减最大值防溢出）；T 为温度"""
    z = m / T - m.max(axis=1, keepdims=True) if T != 1.0 else m - m.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

print("NumPy", np.__version__, "| 中文字体:", _cjk)

## 实验一：one-hot vs 语义向量——"相似度"这个概念是怎么诞生的

one-hot 的三宗罪之一：**任意两个不同词的点积恒为 0**——"猫"和"狗"、"猫"和"汽车"在数学上完全平等。
换成正文 01 页那个 2 维玩具语义世界（国王/王后/男/女），同样的点积立刻有了结构，
连"国王 − 男 + 女 ≈ 王后"的语义算术都能复现。

In [ ]:
words = ["猫", "狗", "汽车", "苹果", "香蕉"]
onehot = np.eye(len(words))                  # one-hot：词典 5 个词，每个词一个单位向量
sim_onehot = onehot @ onehot.T               # 两两点积 → 相似度矩阵
print("one-hot 的点积相似度矩阵（1=自己，0=其它）：")
print("       " + "  ".join(words))
for w, row in zip(words, sim_onehot):
    print(f"{w}  " + "  ".join(f"{v:.0f}" for v in row))
print("→ 除对角线外全是 0：one-hot 表示里'相似度'这个概念直接失效。\n")

# 语义向量：正文 01 页的 2 维玩具世界（第 1 维≈权力，第 2 维≈性别女）
sem_words = ["国王", "王后", "男", "女"]
sem_vecs = np.array([[0.90, 0.10], [0.86, 0.48], [0.10, -0.40], [0.05, 0.00]])
normed = sem_vecs / np.linalg.norm(sem_vecs, axis=1, keepdims=True)
sim_sem = normed @ normed.T                  # 余弦相似度矩阵（归一化后点积=余弦）
print("语义向量的余弦相似度矩阵：")
print("       " + "  ".join(sem_words))
for w, row in zip(sem_words, sim_sem):
    print(f"{w}  " + "  ".join(f"{v:+.2f}" for v in row))

target = sem_vecs[0] - sem_vecs[2] + sem_vecs[3]   # 语义算术：国王 − 男 + 女
dists = np.linalg.norm(sem_vecs - target, axis=1)
rank = dists.argsort()
print(f"\n国王 − 男 + 女 = [{target[0]:.2f}, {target[1]:.2f}]")
print(f"最近邻 = {sem_words[rank[0]]}（距离 {dists[rank[0]]:.3f}）；"
      f"第 2 名 {sem_words[rank[1]]}（距离 {dists[rank[1]]:.3f}，远了 {dists[rank[1]]/dists[rank[0]]:.0f} 倍）")
# 预期输出：one-hot 矩阵=单位阵；语义矩阵里 国王-王后 +0.92、男-女 −0.26——结构出现了；
#           国王−男+女 的最近邻是王后（距离 0.022，正文手算同款），第 2 名才是国王本人

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(sem_vecs[:, 0], sem_vecs[:, 1], s=90, color="tab:blue")
for w, v in zip(sem_words, sem_vecs):
    ax.annotate(w, v + [0.02, 0.03], fontsize=13)
ax.scatter(*target, s=120, marker="*", color="tab:red", label="国王 − 男 + 女")
ax.annotate("国王−男+女", target + [0.02, -0.07], fontsize=11, color="tab:red")
ax.annotate("", xy=target, xytext=sem_vecs[0],
            arrowprops=dict(arrowstyle="->", color="gray", ls="--"))
ax.set_title("2 维玩具语义空间：语义算术走到王后附近")
ax.set_xlabel("第 1 维（≈权力）"); ax.set_ylabel("第 2 维（≈性别女）")
ax.set_xlim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()

**观察**：同一套"点积"运算，在 one-hot 上只能输出单位阵（谁和谁都无关），
在语义向量上却给出结构：国王–王后 **+0.92**、男–女 **−0.26**（方向不同的人反而负相关）。
"国王 − 男 + 女"的真实最近邻是**王后**（距离 0.022，与正文手算一致），第 2 名才是国王本人（0.403）。
词向量的贡献不是"算相似度"这个动作，而是让**相似度有内容可算**——把"观其伴，知其义"压缩进几何。

## 实验二：softmax + 交叉熵的梯度恒等式 ∂L/∂z = p − y

正文 02 页"最美的恒等式"：softmax 嫁给交叉熵后，梯度就是**当前概率 − 应有概率**。
先用正文同款数字（z=[2,1,0]、标签第 2 类）手推 + 数值微分双重验证，
再用这条梯度写一个约 20 行的 softmax 逻辑回归，亲眼看它收敛——
第 3 章多分类、本章注意力、LLM 下一词预测、对比学习，用的全是这一条梯度。

In [ ]:
z = np.array([2.0, 1.0, 0.0])                # logits（正文数值验算同款）
p = np.exp(z) / np.exp(z).sum()              # softmax 输出
y = np.array([0.0, 1.0, 0.0])                # 正确类别是第 2 类（one-hot）
grad_exact = p - y                           # 恒等式：∂L/∂z = p − y

def cross_entropy(zz):                       # L = −Σ y·log softmax(z)
    q = np.exp(zz) / np.exp(zz).sum()
    return -np.sum(y * np.log(q))

eps = 1e-6                                   # 数值微分：中心差分逐维验证
grad_num = np.array([(cross_entropy(z + eps * np.eye(3)[j])
                      - cross_entropy(z - eps * np.eye(3)[j])) / (2 * eps)
                     for j in range(3)])
print("softmax 输出 p      =", p.round(3), "（正文验算值 [0.665, 0.245, 0.090]）")
print("恒等式梯度 p − y    =", grad_exact.round(3))
print("数值微分梯度        =", grad_num.round(3))
print("两者最大差异        = %.2e（≈0：恒等式成立）" % np.abs(grad_exact - grad_num).max())
# 预期输出：p−y = [0.665, −0.755, 0.090]——第 1 类"给多了"被往下压、
#           第 2 类"给少了"被往上拉；各分量之和 = Σp − Σy = 0，方向永远躺在概率单纯形内

In [ ]:
from sklearn.datasets import make_blobs

Xc, yc = make_blobs(n_samples=300, centers=[[-2, -2], [2, -2], [0, 2]],
                    cluster_std=1.2, random_state=0)   # 3 类 2 维合成数据
W = np.zeros((2, 3)); b = np.zeros(3)                  # 待学参数：线性层 W·x + b
lr, history = 0.5, []
for epoch in range(200):                               # 约 20 行的 softmax 回归
    logits = Xc @ W + b
    prob = softmax_rows(logits)                        # [300, 3]
    loss = -np.log(prob[np.arange(len(yc)), yc]).mean()
    history.append(loss)
    g = (prob - np.eye(3)[yc]) / len(yc)               # 整批平均梯度：还是 p − y
    W -= lr * Xc.T @ g                                 # 链式法则：∂L/∂W = Xᵀ(p−y)
    b -= lr * g.sum(axis=0)
acc = ((Xc @ W + b).argmax(axis=1) == yc).mean()
print("最终 loss = %.4f，训练准确率 = %.1f%%" % (history[-1], acc * 100))

plt.figure(figsize=(6.5, 3.5))
plt.plot(history)
plt.title("梯度 p−y 驱动的 softmax 逻辑回归：交叉熵损失单调下降")
plt.xlabel("训练轮次"); plt.ylabel("交叉熵损失")
plt.tight_layout(); plt.show()
# 预期输出：loss 从 ~1.1 一路降到 ~0.2 以下，训练准确率 90% 上下——
#           没有框架、没有自动求导，一条 p−y 就把 3 分类训收敛了

**观察**：数值微分与 p−y 逐位一致（差异 ~1e-11 量级）；
那条"预测减答案"的梯度把 300 个样本的 3 分类干净利落地训到收敛。
LLM 训练时对 logits 求梯度用的还是它——只是 y 换成了"下一个词的 one-hot"。

## 实验三：手写注意力（numpy）——QKV、√d 缩放、热力图与温度

正文 02 页动手试试的 numpy 版：3 词句子 ["我", "爱", "猫"]，
投影出 Q/K/V → 打分 QKᵀ/√d → 逐行 softmax → 加权混合 V，全程矩阵乘法。
顺带验证"点积方差 = d、除以 √d 拉回 1"的缩放推导，再看温度怎么改变权重分布。
想拖参数、换句子看颜色变化，请玩本库实验场 [attention.html](../playground/attention.html)。

In [ ]:
rng = np.random.default_rng(0)
words3 = ["我", "爱", "猫"]
n, d = 3, 8                                  # 序列长 3、维度 8（真实模型 4096 量级）
E = rng.normal(0, 1, (n, d))                 # 词向量（真实模型来自词表 embedding 查表）
Wq = rng.normal(0, 1, (d, d)) / np.sqrt(d)   # 三个投影矩阵（Xavier 式缩放，标准做法）
Wk = rng.normal(0, 1, (d, d)) / np.sqrt(d)
Wv = rng.normal(0, 1, (d, d)) / np.sqrt(d)

Q, K, V = E @ Wq, E @ Wk, E @ Wv             # 同一个 X 乘三个矩阵 → 三份"视角"

# √d 缩放验证：用 2000 个"词"的大批量算打分标准差（3×3 太小、抽样噪声大）
Eb = rng.normal(0, 1, (2000, d))
raw_big = (Eb @ Wq) @ (Eb @ Wk).T
print("大批量打分：缩放前标准差 = %.2f（理论 √d = %.2f）" % (raw_big.std(), np.sqrt(d)))
print("            除以 √d 后  = %.2f（拉回 1 量级，softmax 才不会饱和成 one-hot）\n"
      % (raw_big.std() / np.sqrt(d)))

raw = Q @ K.T                                # 打分表 [n,n]
scores = raw / np.sqrt(d)                    # √d 缩放：给分数降温
A = softmax_rows(scores)                     # 逐行 softmax → 注意力权重
out = A @ V                                  # 加权混合 V：输出 [n,d]，与输入同构
print("注意力权重矩阵（行 = 查询词 Q，列 = 被看词 K）：")
print("      " + "  ".join(words3))
for w, row in zip(words3, A):
    print(f"{w}   " + "  ".join(f"{v:.2f}" for v in row))
print("每行的和 =", A.sum(axis=1).round(4), "——softmax 保证蛋糕总量恒为 1")
print("输出形状 =", out.shape, "（与输入同构 → 可以一层层堆下去）\n")

print("温度怎么改权重（把打分除以 T 再 softmax，T 越小分布越尖）：")
for T in [0.5, 1.0, 2.0]:
    At = softmax_rows(scores / T)
    spread = (At.max(axis=1) - At.min(axis=1)).mean()
    print(f"T={T}: 『爱』行 = {At[1].round(3)}，平均极差 = {spread:.3f}")
# 预期输出：缩放前后标准差 ≈ √8 : 1 ≈ 2.8 : 1；行和恒等于 1；
#           T=0.5 权重极差约为 T=2 的 4 倍——低温削尖、高温摊平

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, T in zip(axes, [0.5, 1.0, 2.0]):
    At = softmax_rows(scores / T)
    im = ax.imshow(At, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3), words3); ax.set_yticks(range(3), words3)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{At[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(f"注意力热力图 T={T}")
    ax.set_xlabel("被看词（K）"); ax.set_ylabel("查询词（Q）")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

**观察**：缩放前后的打分标准差正好差 √d 倍——不缩放的话大维度下 softmax 会饱和成
"一家独大"、梯度趋 0（训不动的根源）。行和恒为 1：注意力是一份"蛋糕切分方案"。
温度是同一枚旋钮：T=0.5 权重极差拉大（聚焦），T=2 权重摊平（稀释）——
与实验五的采样温度、第 8 章双塔的检索温度是同一个数学对象。

## 实验四：多头 = 切分，不是复制——d=4 切 2 头的对照

高频误解：多头不是把向量复制 h 份各算一次，而是把 $d_{model}$ **切成 h 份、
每头 $d_k = d/h$ 维、各自在子空间里做一次小注意力再拼接**——参数量与单头满维基本相同。
用 d=4、h=2 打印三张注意力图对照：单头（4 维整空间）vs 头 1 / 头 2（各 2 维子空间）。

In [ ]:
rng = np.random.default_rng(1)
d, h = 4, 2                                  # d_model=4，切 2 头，每头 d_k=2
dk = d // h
E = rng.normal(0, 1, (3, d))
Wq, Wk, Wv = (rng.normal(0, 1, (d, d)) for _ in range(3))
Q, K, V = E @ Wq, E @ Wk, E @ Wv

A_full = softmax_rows(Q @ K.T / np.sqrt(d))  # 单头：整个 4 维空间一次注意力
heads = []
for i in range(h):                           # 多头：切列，各自在自己的子空间打分
    sl = slice(i * dk, (i + 1) * dk)
    heads.append(softmax_rows(Q[:, sl] @ K[:, sl].T / np.sqrt(dk)))

print("单头（4 维整空间，√4 缩放）：\n", A_full.round(3))
print("头 1（第 1~2 维子空间，√2 缩放）：\n", heads[0].round(3))
print("头 2（第 3~4 维子空间，√2 缩放）：\n", heads[1].round(3))
print("参数量：单头 W = %d×%d = %d；两头 = 2×(%d×%d) = %d —— 切分不增加参数"
      % (d, d, d * d, d, dk, 2 * d * dk))

concat = np.concatenate([heads[0] @ V[:, :dk], heads[1] @ V[:, dk:]], axis=1)
print("两头各自加权混合 V 再拼接，输出形状 =", concat.shape, "——与单头输出同构，可继续堆层")
# 预期输出：三张权重矩阵互不相同——两个头在两个子空间里各学各的"看谁"；
#           参数量 16 = 8 + 8，多头是"免费"的视角拆分

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
mats = [(A_full, "单头：4 维整空间"), (heads[0], "头 1：第 1~2 维"), (heads[1], "头 2：第 3~4 维")]
for ax, (m, ttl) in zip(axes, mats):
    im = ax.imshow(m, cmap="Oranges", vmin=0, vmax=1)
    ax.set_xticks(range(3), words3); ax.set_yticks(range(3), words3)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{m[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(ttl)
    ax.set_xlabel("被看词（K）"); ax.set_ylabel("查询词（Q）")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

**观察**：三张注意力图明显不同——同一个句子，头 1 和头 2 在各自的 2 维子空间里
"看谁"的答案不一样（真实模型里这就是"一组盯语法、一组盯指代"的来源，分工是学出来的）。
参数账：单头 16 = 两头 8+8，切分免费；但把 h 切到 4（每头 1 维）时点积区分度会崩——
**头数是"够用就好"，不是越多越好**。

## 实验五：温度采样——logits=[2, 0.5, −1] 在 T=0.5 / 1 / 2 下的分布

正文"温度的数学"深潜的可跑版：$P_T(i) = \mathrm{softmax}(z_i/T)$。
低温"富者愈富"（T→0 退化为 argmax 贪心）、高温"均贫富"（T→∞ 趋近均匀），
任意两候选的几率比只被温度按比例缩放：$P_T(i)/P_T(j) = e^{(z_i-z_j)/T}$。
用 2000 次真实采样验证理论分布。

In [ ]:
logits = np.array([2.0, 0.5, -1.0])
tokens = ["好", "不错", "冷"]                # 假装这是"今天天气真___"的下注表前 3 名

def softmax_T(z, T):
    zz = z / T
    e = np.exp(zz - zz.max())
    return e / e.sum()

print("T     P(好)   P(不错)  P(冷)    前两名几率比（理论 e^(1.5/T)）")
for T in [0.5, 1.0, 2.0]:
    pr = softmax_T(logits, T)
    print(f"{T:.1f}   {pr[0]:.3f}   {pr[1]:.3f}   {pr[2]:.3f}    "
          f"{pr[0]/pr[1]:5.1f}:1（{np.exp(1.5/T):.1f}:1）")
# 预期输出：T=0.5 → [0.950, 0.047, 0.002]，几率比 20:1（赢家通吃，接近贪心）
#           T=1.0 → [0.786, 0.175, 0.039]，几率比 4.5:1
#           T=2.0 → [0.590, 0.279, 0.132]，几率比 2.1:1（均贫富，接近均匀 1/3）

In [ ]:
rng = np.random.default_rng(3)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, T in zip(axes, [0.5, 1.0, 2.0]):
    pr = softmax_T(logits, T)
    samples = rng.choice(3, size=2000, p=pr)          # 按分布真实采样 2000 次
    freq = np.bincount(samples, minlength=3) / 2000   # 采样频率
    ax.bar(np.arange(3) - 0.2, pr, width=0.4, label="理论概率", color="tab:blue")
    ax.bar(np.arange(3) + 0.2, freq, width=0.4, label="采样频率(2000次)", color="tab:orange")
    ax.set_xticks(range(3), tokens)
    ax.set_title(f"温度 T={T}：低温削尖 / 高温摊平")
    ax.set_ylabel("概率"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
# 预期输出：三组柱子里"理论概率"与"采样频率"几乎重合——大数定律兑现了分布；
#           T=0.5 时"冷"几乎抽不到，T=2 时三家相差无几

**观察**：理论概率与 2000 次采样频率吻合；几率比严格等于 $e^{\Delta z/T}$
（20:1 → 4.5:1 → 2.1:1）。注意温度**不改变排名**——"好"永远是第一，
温度只调"敢不敢冒险"；模型把错词排在第一时，降温只会让错误更斩钉截铁（幻觉更自信）。

## 实验六：真实大模型生成（transformers）——需本地环境 + 联网

正文 04 页动手试试的同款代码：GPT-2 级小模型演示"下一词接龙"、贪心 vs 高温、低温复读现场。
**本 cell 需 `pip install transformers torch`，且首次运行要联网下载模型（约 500MB）**；
本仓库的离线验证环境未执行此 cell（仅语法校验），预期输出来自正文同款代码的运行经验。

In [ ]:
# 依赖：pip install transformers torch（需本地环境 + 首次联网下载 GPT-2，约 500MB）
# ⚠️ 离线验证环境未执行本 cell，仅保证语法正确；装好依赖后可直接运行
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")   # GPT-2 级小模型：只会续写、无对话对齐

# 1) 贪心解码：永远挑概率最高的词（零温极限），输出固定
print(generator("The weather today is really",
                max_new_tokens=8, do_sample=False)[0]["generated_text"])
# 预期输出：每次运行完全相同，如 "...really nice and sunny..."——贪心 = argmax

# 2) 高温采样：T=1.5，冷门词上位，每次运行结果都可能不同
print(generator("The weather today is really",
                max_new_tokens=8, do_sample=True, temperature=1.5)[0]["generated_text"])
# 预期输出：明显更"放飞"——对应实验五里 T=2 那张被摊平的分布

# 3) 低温复读现场 + 重复惩罚急救
print(generator("The list of things I like: apples, apples,",
                max_new_tokens=20, do_sample=False, repetition_penalty=1.0)[0]["generated_text"])
# 预期输出：多跑几次观察结尾是否陷入重复片段（低温复读）；
#           把 repetition_penalty 改成 1.2 重跑，复读明显缓解——三个旋钮一次只动一个

**预期观察**（本地运行后）：贪心版每次输出相同；高温版每次不同且更放飞；
长生成的贪心路径容易陷入重复片段——"低温复读"的现场，重复惩罚是急救药。
这个小模型没学过多少中文，所以示例用英文；对话能力来自下一页的微调与对齐，
预训练本身只有接龙。生成机制的数学（温度/几率比）已在实验五离线验证。

## 实验七：最小 RAG 检索（numpy）——手造语义向量 + top-2 命中 + 阈值拒答

正文 06 页 RAG 的"检索"这一步：文档块 → 向量建库，问题 → 向量，余弦相似度取 top-k。
真实的 embedding 模型需联网下载（sentence-transformers，约 100~500MB），
这里用**6 维可解释的手造语义向量**复刻整套机制——每个维度就是一个语义主题，
检索、top-2、阈值拒答（"库里没有的别硬答"）全部离线可跑。

In [ ]:
docs = [
    "报销制度：差旅发票需在 30 天内提交，单程超过 800 元需提前审批。",
    "休假制度：年假按工龄 5～15 天，入职满 1 年起享，可跨年结转 1 次。",
    "考勤制度：工作日 9:00-18:00，每月迟到 3 次以内口头提醒。",
    "加班制度：加班需提前申请，调休按 1:1 结转，月底清零。",
    "工资制度：每月 10 日发薪，绩效工资按季度核算。",
]
dims = ["报销", "休假", "考勤", "加班", "工资", "病假"]     # 6 个语义维度（真实模型是几百维的黑盒）
D = np.array([                                              # 每段文档的手造向量（行=文档）
    [0.95, 0.00, 0.00, 0.10, 0.10, 0.00],   # 报销制度
    [0.00, 0.95, 0.05, 0.10, 0.00, 0.00],   # 休假制度
    [0.00, 0.05, 0.95, 0.20, 0.00, 0.00],   # 考勤制度
    [0.05, 0.15, 0.20, 0.95, 0.10, 0.00],   # 加班制度
    [0.10, 0.00, 0.00, 0.05, 0.95, 0.00],   # 工资制度
])
print("语料 × 语义维度 的向量库：")
print("        " + "  ".join(dims))
for s, v in zip(["报销", "休假", "考勤", "加班", "工资"], D):
    print(f"{s}制度  " + "  ".join(f"{x:4.2f}" for x in v))

# 查询向量：关键词命中哪个语义维，就往哪维加分（真实系统由 embedding 模型完成）
kw = {"报销": ("报销", 1.0), "发票": ("报销", 0.9), "审批": ("报销", 0.6),
      "年假": ("休假", 0.9), "休假": ("休假", 0.9), "请假": ("休假", 0.8),
      "迟到": ("考勤", 0.9), "打卡": ("考勤", 0.9), "上班": ("考勤", 0.6),
      "加班": ("加班", 1.0), "调休": ("加班", 0.9),
      "工资": ("工资", 1.0), "薪水": ("工资", 0.9), "发薪": ("工资", 0.9),
      "病假": ("病假", 1.0)}

def embed_query(q):
    v = np.zeros(len(dims))
    for w, (dim, wgt) in kw.items():                        # 命中关键词 → 给对应语义维加分
        if w in q:
            v[dims.index(dim)] += wgt
    return v

def normalize(m):
    return m / (np.linalg.norm(m, axis=-1, keepdims=True) + 1e-9)

Dn = normalize(D)                                           # 建库：文档向量归一化
THRESHOLD = 0.75                                            # 拒答阈值：最高分低于它就不硬答
for q in ["年假有几天？", "发票超过 800 元怎么报销？", "病假工资怎么算？"]:
    qv = normalize(embed_query(q))
    scores = Dn @ qv                                        # 检索：余弦相似度（归一化后点积）
    order = np.argsort(-scores)
    print(f"\n[{q}] 各块得分：{scores.round(2).tolist()}")
    if scores[order[0]] < THRESHOLD:                        # 阈值拒答：组装提示词之前先拦一道
        print(f"→ 最高相似度 {scores[order[0]]:.2f} 低于阈值 {THRESHOLD}：拒答"
              f"（制度库中暂无相关规定）")
    else:
        top2 = [(docs[i], scores[i]) for i in order[:2]]
        print(f"→ 命中 top-2：1) {top2[0][0][:12]}…（{top2[0][1]:.2f}）"
              f"  2) {top2[1][0][:12]}…（{top2[1][1]:.2f}）")
        print(f"→ 组装提示词：只根据以下材料回答：[{top2[0][0]}] 问题：{q}（交给任意 LLM 生成）")
# 预期输出：前两问正常命中（休假/报销，得分 0.9+）；
#           第三问"病假工资"最高分 ~0.70 < 0.75 → 拒答——库里没有病假规定，检索替模型挡了一次幻觉

**观察**："年假"与"发票报销"都精准命中 top-1（得分 0.99，第二名被甩到 0.15）；
而"病假工资怎么算"虽然蹭到了"工资"维度，最高分仍只有 0.70 < 0.75 → **拒答**。
这正是正文 RAG 失败案例的现场：没有阈值时，这个问题会把"工资制度"块硬塞给模型——
检索层的"知道自己不知道"，是 RAG 对抗幻觉的第一道闸。真实系统把"手造向量"
换成 embedding 模型（sentence-transformers，需联网下载），机制一模一样。

## 改参数建议（一次只改一个，观察一个）

1. **实验一**：把"女"的向量改成 [0.05, 0.60]，看"国王−男+女"的最近邻怎么漂——玩具数字是精心构造的，真实词向量里语义算术只是**近似**成立；
2. **实验二**：把交叉熵换成均方误差 `(p − y)**2` 的平方损失重写梯度——多出一堆交叉项、还容易饱和，体会"softmax+交叉熵是 天作之合"这句话的分量；
3. **实验三**：删掉 `/ np.sqrt(d)` 再把 d 改成 64、512，看每行最大权重如何逼近 1（softmax 饱和 = 梯度消失现场）；把 n 改成 2000 看权重被摊薄（注意力稀释）；
4. **实验四**：头数 h 改成 4（每头只剩 1 维），看两张权重图是否变得难以区分——"头数越多越好"为什么不成立；
5. **实验五/七**：给采样加 top-k=2 截断，看 T=2 时冷门词"冷"是否彻底出局（温度与截断正交）；实验七把 THRESHOLD 降到 0.5，"病假工资"被硬答——体会"宁可少答 vs 宁可错答"的旋钮必须按自己的向量空间重新标定。